# Bonus: RAG — Retrieval-Augmented Generation

## What you will learn in this notebook

- What **RAG** is and the problem it solves
- How to **load and chunk** a PDF into searchable pieces
- What **embeddings** are and why they enable semantic search
- How to build and query an **in-memory vector store**
- How to wrap vector search as a **tool** so an agent can use it

---

### The problem: private knowledge

LLMs are trained on public internet data. They know nothing about:
- Your company's internal documents
- Your employee handbook
- Your product docs, contracts, research papers
- Any private or recent document

You could put the entire document in the system prompt — but a 200-page handbook would use millions of tokens per call. Too slow and too expensive.

**RAG** solves this by searching for *only the relevant passages* and sending just those to the model.

### How RAG works

```
INDEXING (done once, upfront):
  Document → split into chunks → embed each chunk → store in vector DB

RETRIEVAL (done per query):
  User question → embed question → find most similar chunks → return top-k

GENERATION (LLM step):
  LLM reads: [top-k chunks] + [user question] → generates grounded answer
```

### What is an embedding?

An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of text. Texts with similar meanings have vectors that are close together in high-dimensional space.

```
"How many vacation days do I get?"  →  [0.12, -0.34, 0.87, ...]  (1536 numbers)
"Annual leave entitlement policy"   →  [0.14, -0.31, 0.85, ...]  (very similar!)
"The company uses Python"           →  [-0.43, 0.22, -0.15, ...]  (very different)
```

A vector store finds the chunks whose vectors are **closest** to the question's vector — this is **semantic search** (meaning-based), not keyword search.

In [ ]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================
# Requires OPENAI_API_KEY — used for both the embedding model
# (text-embedding-3-large) and the chat model (gpt-5-nano).

from dotenv import load_dotenv

load_dotenv()

---
## Section 1 — Semantic Search Pipeline

### Step 1: Load → Step 2: Split → Step 3: Embed → Step 4: Store → Step 5: Search

In [ ]:
# ============================================================
# CELL 2: Step 1 — Load the PDF
# ============================================================
# PyPDFLoader reads a PDF file and converts it into a list of
# LangChain Document objects — one per page.
#
# Each Document has:
#   .page_content  → the text extracted from that page
#   .metadata      → dict with 'source' (file path) and 'page' (page number)
#
# The metadata is preserved through chunking and stored in the
# vector store — so when we retrieve chunks, we know which page
# and file they came from (useful for citations).
#
# PyPDFLoader requires: pip install pypdf

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")

data = loader.load()  # Returns list of Document objects, one per page

print(data)  # Preview the loaded pages

In [ ]:
# ============================================================
# CELL 3: Step 2 — Split Into Chunks
# ============================================================
# We can't embed entire pages — they may be too long and a single
# embedding would average out the meaning of many topics.
# We split each page into smaller overlapping chunks.
#
# RecursiveCharacterTextSplitter parameters:
#
#   chunk_size=1000
#     → Each chunk is at most 1000 characters.
#     → Small enough to be topically focused, large enough to have context.
#     → Rule of thumb: 500-1500 chars is a good range for most documents.
#
#   chunk_overlap=200
#     → Consecutive chunks share 200 characters.
#     → Prevents important sentences from being split across chunks
#       with no context on either side.
#     → Typically 10-20% of chunk_size.
#
#   add_start_index=True
#     → Stores the character offset in metadata — useful for
#       locating the chunk in the original document.
#
# "Recursive" in the name means it tries to split on paragraphs first,
# then sentences, then words — preserving natural boundaries.

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,         # Max characters per chunk
    chunk_overlap=200,       # Overlap between consecutive chunks
    add_start_index=True     # Record chunk position in metadata
)

all_splits = text_splitter.split_documents(data)

print(len(all_splits))  # Total number of chunks — each will get its own embedding

Embedding model options: https://docs.langchain.com/oss/python/integrations/text_embedding

In [ ]:
# ============================================================
# CELL 4: Step 3 — Create an Embedding Model
# ============================================================
# An embedding model converts text → a vector of numbers.
#
# OpenAIEmbeddings uses OpenAI's embedding API:
#   model="text-embedding-3-large"
#   → Produces 3072-dimensional vectors
#   → Higher dimensional = more nuanced meaning representation
#   → Also available: "text-embedding-3-small" (1536-dim, cheaper)
#
# The embedding model is used TWICE:
#   1. During indexing: embed each chunk (done once, upfront)
#   2. During retrieval: embed each query (done per question)
#
# Both must use the SAME model — mixing models gives wrong distances.
#
# Other options (see link above):
#   HuggingFaceEmbeddings  → free, runs locally, no API key needed
#   CohereEmbeddings       → good multilingual support
#   GoogleGenerativeAIEmbeddings → Google's embedding models

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
# ============================================================
# CELL 5: Step 4a — Create an In-Memory Vector Store
# ============================================================
# A vector store is a database optimised for similarity search.
# Given a query vector, it quickly finds the k most similar stored vectors.
#
# InMemoryVectorStore:
#   → Stores vectors in RAM (Python dict)
#   → Perfect for demos and small document sets
#   → Lost on process restart
#   → No installation needed
#
# For production, swap to:
#   Pinecone     → managed cloud vector DB (what we use in the course's RAG pipeline)
#   pgvector     → PostgreSQL extension — embed vector search in your existing DB
#   Chroma       → open-source, easy local setup
#   FAISS        → Facebook's library, fast for large local datasets
#   Weaviate     → open-source with rich filtering capabilities
#
# All of these implement the same LangChain VectorStore interface:
#   .add_documents(...)  and  .similarity_search(...)
# Swapping providers is usually one line change.

from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)  # Pass the embedding model

In [ ]:
# ============================================================
# CELL 6: Step 4b — Embed and Store All Chunks
# ============================================================
# add_documents() does two things for each chunk:
#   1. Calls the embedding model API to get the vector
#   2. Stores (vector, document) in the vector store
#
# This makes ONE API call per chunk — so if you have 100 chunks,
# it makes 100 embedding API calls. For large documents, this can
# take time and cost money. In production you do this ONCE and
# persist the vector store (to Pinecone, pgvector, etc.).
#
# ids = list of document IDs assigned to each chunk.
# Useful if you later want to update or delete specific chunks.

ids = vector_store.add_documents(documents=all_splits)
# Now the vector store has embeddings for all chunks — ready to search

In [ ]:
# ============================================================
# CELL 7: Step 5 — Test Semantic Search Directly
# ============================================================
# similarity_search(query) does:
#   1. Embeds the query string using the same embedding model
#   2. Computes cosine similarity between query vector and all stored vectors
#   3. Returns the top-k most similar Documents (default k=4)
#
# Notice the question is phrased naturally — NOT as a keyword search.
# The embedding captures the semantic meaning (vacation days, first year)
# and finds chunks about annual leave policy — even if they don't
# use the exact words in the query.
#
# results[0] → the most relevant chunk
# .page_content → the text of that chunk
# .metadata     → includes page number and source file

results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])  # Most relevant chunk — should mention vacation/leave policy

---
## Section 2 — RAG Agent

Now we wrap the vector search in a tool so an agent can use it automatically.

In [ ]:
# ============================================================
# CELL 8: Wrap Vector Search as a Tool
# ============================================================
# The tool receives a natural language query from the agent,
# runs semantic search against the vector store, and returns
# the most relevant passage.
#
# Design decision: return only results[0].page_content
#   → The single most relevant chunk, as plain text
#   → Simple and cheap — the model gets focused context
#   → Alternative: return all k results joined by \n for broader coverage
#
# The docstring "Search the employee handbook" is crucial —
# it tells the agent when to use this tool vs others.
#
# In production you might also return metadata (page number, section)
# so the agent can cite its sources in the response.

from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)  # Find relevant chunks
    return results[0].page_content                   # Return the best match

In [ ]:
# ============================================================
# CELL 9: Create the RAG Agent
# ============================================================
# A RAG agent is just a regular agent with a search tool.
# The system prompt scopes it to the handbook — preventing it
# from answering questions from general knowledge (which would
# bypass the RAG pipeline and risk inaccurate answers).
#
# "search the employee handbook for information"
#   → Explicit instruction to always use the tool
#   → Without this, the model might answer general HR questions
#     from training data instead of the actual handbook

from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information."
)

In [ ]:
# ============================================================
# CELL 10: Ask a Question About the Handbook
# ============================================================
# Full RAG pipeline triggered by this one call:
#
#   1. Agent reads the question
#   2. Decides to call search_handbook
#   3. search_handbook embeds the query → similarity search → returns chunk
#   4. Agent reads the chunk
#   5. Agent generates an answer grounded in the actual handbook text
#
# The answer comes from the handbook, not from the model's training data.
# This is the core RAG guarantee: grounded, verifiable answers.

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}
)

In [ ]:
# ============================================================
# CELL 11: Inspect the Full Trace
# ============================================================
# Look at messages[2] (ToolMessage) to see the raw chunk that was
# retrieved from the handbook. Then look at messages[3] (AIMessage)
# to see how the model used it to compose its answer.
#
# This trace is your ground truth for debugging RAG:
#   - Was the right chunk retrieved? (check messages[2].content)
#   - Did the model correctly interpret the chunk? (check messages[3].content)
#   - If retrieval was wrong → adjust chunk_size or embedding model
#   - If interpretation was wrong → adjust system prompt or add more chunks

from pprint import pprint

pprint(response)

---
## Summary

| Concept | Key takeaway |
|---|---|
| **RAG** | Index private docs once → search for relevant chunks per query → LLM generates grounded answer |
| **PyPDFLoader** | Loads a PDF into one Document per page, preserving page metadata |
| **RecursiveCharacterTextSplitter** | Splits docs into overlapping chunks, respecting natural boundaries |
| **Embeddings** | Vectors representing semantic meaning — similar texts have similar vectors |
| **`chunk_size`** | Max chars per chunk — smaller = more focused, larger = more context |
| **`chunk_overlap`** | Shared chars between chunks — prevents information loss at boundaries |
| **`similarity_search(query)`** | Embed query → find k nearest chunks in vector space |
| **`search_handbook` tool** | Wraps vector search so the agent can call it on demand |

### The RAG pipeline (memorise this)

```
INDEXING (once):  PDF → pages → chunks → embeddings → vector store
QUERYING (each):  question → embedding → similar chunks → LLM → answer
```

### Scaling to production

Swap `InMemoryVectorStore` for `Pinecone` or `pgvector`:
- Survives restarts
- Handles millions of chunks
- Supports metadata filtering (only search this department's docs)
- The `.add_documents()` and `.similarity_search()` interface is identical